Data obtained here: https://zenodo.org/records/14767363  
egrid details here: https://www.epa.gov/system/files/documents/2025-01/egrid2023_technical_guide.pdf  
ejscreen in action here: https://pedp-ejscreen.azurewebsites.net/  
ejscreen documentation here: https://www.epa.gov/system/files/documents/2024-07/ejscreen-tech-doc-version-2-3.pdf  
other resource i couldnt figure out: https://dataverse.harvard.edu/dataset.xhtml?persistentId=doi:10.7910/DVN/RLR5AX  
ejscreen tool archive: https://screening-tools.com/epa-ejscreen  


In [1]:
import pandas as pd
import numpy as np

In [2]:
import requests
# ref https://www.geeksforgeeks.org/python/how-to-download-files-from-urls-with-python/#

#This is census data for each block group id, it gives the mean latitude and longitude
url = 'https://www2.census.gov/geo/docs/reference/cenpop2020/blkgrp/CenPop2020_Mean_BG.txt'

response = requests.get(url)
file_path = 'data/CenPop2020_Mean_BG.txt'

if response.status_code == 200:
    with open(file_path, 'wb') as file:
        file.write(response.content)
    print('File downloaded successfully')
else:
    print('Failed to download file')

File downloaded successfully


In [3]:
#block group means
mean_bg = pd.read_csv(file_path)

mean_bg['STATEFP'] = mean_bg['STATEFP'].astype(str).str.zfill(2)
mean_bg['COUNTYFP'] = mean_bg['COUNTYFP'].astype(str).str.zfill(3)
mean_bg['TRACTCE'] = mean_bg['TRACTCE'].astype(str).str.zfill(6)
mean_bg['BLKGRPCE'] = mean_bg['BLKGRPCE'].astype(str).str.zfill(1)
mean_bg['ID'] = mean_bg['STATEFP']+ mean_bg['COUNTYFP'] + mean_bg['TRACTCE'] + mean_bg['BLKGRPCE']
#rad needed for efficient nearest plant finding
mean_bg['LAT_RAD'] = np.deg2rad(mean_bg['LATITUDE'])
mean_bg['LON_RAD'] = np.deg2rad(mean_bg['LONGITUDE'])

mean_bg = mean_bg[['ID', 'LAT_RAD', 'LON_RAD']].set_index('ID')
mean_bg

,LAT_RAD,LON_RAD
ID,,
010010201001,0.566612,-1.509471
010010201002,0.566931,-1.509478
010010202001,0.566854,-1.509264
010010202002,0.566640,-1.509202
010010203001,0.566846,-1.509009
...,...,...
721537506011,0.314508,-1.166594
721537506012,0.314512,-1.166662
721537506013,0.314437,-1.166711


In [4]:
use_cols = [
    'ID', 
    'REGION',
    'PEOPCOLOR', 
    'ACSTOTPOP', #Total population
    'LOWINCOME', 'ACSIPOVBAS', #Population for whom poverty status is determined
    'UNEMPLOYED',  'ACSUNEMPBAS', #Unemployment base--persons in civilian labor force (unemployment rate)
    'LINGISO', #Limited English speaking households
    'ACSTOTHH', #Households (for limited English speaking)
    'LESSHS', 
    'ACSEDUCBAS', #Population 25 years and over (use for less than hs)
    'UNDER5', 
    'OVER64',
    'P_LIFEEXPPCT', 
    'ST_ABBREV', 
    'CNTY_NAME'
]

cols_agg = {
    'ST_ABBREV': 'first',
    'REGION': 'first', 
    'CNTY_NAME':'first',
    'PEOPCOLOR': 'sum',
    'ACSTOTPOP': 'sum',
    'LOWINCOME': 'sum',
    'ACSIPOVBAS': 'sum',
    'UNEMPLOYED': 'sum',
    'ACSUNEMPBAS': 'sum',
    'LINGISO':'sum', 
    'ACSTOTHH': 'sum',
    'LESSHS': 'sum', 
    'ACSEDUCBAS': 'sum',
    'UNDER5': 'sum', 
    'OVER64': 'sum',
    
}

In [5]:
from sklearn.neighbors import BallTree

egrid = pd.read_csv('data/egrid_counties.csv', index_col = 0)
#put plant locations points into radian tuples for balltree
egrid['LAT_RAD'] = np.deg2rad(egrid['Plant latitude'])
egrid['LON_RAD'] = np.deg2rad(egrid['Plant longitude'])
plant_locs = np.array(egrid[['LAT_RAD', 'LON_RAD']])

output = pd.DataFrame()
df = pd.read_csv('data/EJSCREEN_2023_BG_with_AS_CNMI_GU_VI.csv',  usecols = use_cols, encoding = 'latin1', chunksize = 10000)

#BallTree ref: https://autogis-site.readthedocs.io/en/2021/notebooks/L3/06_nearest-neighbor-faster.html#:~:text=While%20Shapely's%20nearest_points%20%2Dfunction%20provides,terms%20of%20supported%20distance%20metrics.
#scikit-learn’s haversine distance metric wants inputs as radians and also outputs the data as radians. 
tree = BallTree(plant_locs, leaf_size=15, metric='haversine')
num_blocks_dropped = 0
for chunk in df:

    chunk['ID'] = chunk['ID'].astype(str).str.zfill(12)
    #get latitude and longitude in radians for each census block in chunk
    chunk = chunk.merge(mean_bg, how='left', left_on='ID', right_index=True)
    num_blocks_dropped += chunk[['LAT_RAD', 'LON_RAD']].isnull().any(axis=1).sum()
    chunk = chunk.dropna(subset=['LAT_RAD', 'LON_RAD'])
  
    block_locs = chunk[['LAT_RAD', 'LON_RAD']]
   
    #find nearest plant for each block group center, distance = dist from block to closest plant (nearest neighbor)
    distance, indice = tree.query(block_locs, k=1)
   
    # Convert to miles from radians where 3959 is earths mean radius in miles
    chunk['DIST_TO_PLANT'] = distance * 3958.8  
    
    # non host communities defined to be > 3 miles and < 50 miles away from a plant
    chunk = chunk[chunk['DIST_TO_PLANT'] > 3]
    chunk = chunk[chunk['DIST_TO_PLANT'] < 50 ]
  
    if not chunk.empty:
        chunk['County FIPS'] = chunk['ID'].astype(str).str.zfill(12).str[:5] #fill front with 0s in case fips codes should be 12 digits
        chunk_grouped = chunk.groupby('County FIPS').agg(cols_agg) #group within chunk for efficiency, but will need to repeat at end
        output = pd.concat([chunk_grouped, output])
    
    #display(output)
    #break
#output
final_df = output.groupby(output.index).agg(cols_agg).reset_index()
display(final_df)
print(f'Out of over 240k block groups, only {num_blocks_dropped} were not dropped from missing latitude and longitude')

,County FIPS,ST_ABBREV,REGION,CNTY_NAME,PEOPCOLOR,ACSTOTPOP,LOWINCOME,ACSIPOVBAS,UNEMPLOYED,ACSUNEMPBAS,LINGISO,ACSTOTHH,LESSHS,ACSEDUCBAS,UNDER5,OVER64
0,01001,AL,4,Autauga County,15668,58239,17782,57790,752,26623,32,21856,4126,39614,3318,8815
1,01003,AL,4,Baldwin County,39583,227131,57840,223772,3994,108361,730,87190,14555,161977,12035,46805
2,01005,AL,4,Barbour County,13991,25259,11195,22250,808,9369,117,9088,4378,17995,1320,4801
3,01007,AL,4,Bibb County,5816,22412,8485,21000,884,9107,23,7083,3125,16057,1196,3594
4,01009,AL,4,Blount County,8289,58884,19518,58323,1554,25798,337,21300,6650,40668,3467,10584
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2574,56037,WY,8,Sweetwater County,9216,42459,9394,41941,1503,22387,307,15529,2029,27816,2657,5390
2575,56039,WY,8,Teton County,4681,23319,5002,23240,368,15320,466,9531,701,17659,1068,3609
2576,56041,WY,8,Uinta County,2661,20514,5206,20267,348,10036,126,7675,851,13233,1413,3022
2577,56043,WY,8,Washakie County,1405,6845,1591,6670,96,3593,10,2934,293,4710,348,1450


Out of over 240k block groups, only 686 were not dropped from missing latitude and longitude


In [6]:
#renaming and calculating to match eGRID naming

# note that need to divide by different column values as specified in EJScreen technical guide
final_df['Total Population'] = final_df['ACSTOTPOP']
final_df['People of Color (%)'] = (final_df['PEOPCOLOR'] / final_df['ACSTOTPOP']) * 100
final_df['Low Income (%)'] = (final_df['LOWINCOME'] / final_df['ACSIPOVBAS']) * 100 #divide by Population for whom poverty status is determined
final_df['Unemployment Rate (%)'] = (final_df['UNEMPLOYED'] / final_df['ACSUNEMPBAS']) * 100 #Unemployment base--persons in civilian labor force (unemployment rate)
final_df['Limited English Speaking (%)'] = (final_df['LINGISO'] / final_df['ACSTOTHH']) * 100 #Households (for limited English speaking)
final_df['Less Than High School Education (%)'] = (final_df['LESSHS'] / final_df['ACSEDUCBAS']) * 100 #Population 25 years and over (use for less than hs)
final_df['Under Age 5 (%)'] = (final_df['UNDER5'] / final_df['ACSTOTPOP']) * 100
final_df['Over Age 64 (%)'] = (final_df['OVER64'] / final_df['ACSTOTPOP']) * 100
final_df['Plant state abbreviation'] = final_df['ST_ABBREV']
final_df['Plant county name'] = final_df['CNTY_NAME']
final_df['has_plant'] = 0
final_df['EPA Region'] = final_df['REGION']


In [7]:
df_drop = final_df.dropna() #only 6 rows dropped
save_df = df_drop[['County FIPS', 'Plant state abbreviation', 'Plant county name', 'EPA Region','has_plant','Total Population', 'People of Color (%)', 'Low Income (%)', 'Less Than High School Education (%)', 'Limited English Speaking (%)', 'Unemployment Rate (%)', 'Under Age 5 (%)', 'Over Age 64 (%)']]
save_df

,County FIPS,Plant state abbreviation,Plant county name,EPA Region,has_plant,Total Population,People of Color (%),Low Income (%),Less Than High School Education (%),Limited English Speaking (%),Unemployment Rate (%),Under Age 5 (%),Over Age 64 (%)
0,01001,AL,Autauga County,4,0,58239,26.902934,30.770029,10.415510,0.146413,2.824625,5.697213,15.135905
1,01003,AL,Baldwin County,4,0,227131,17.427388,25.847738,8.985844,0.837252,3.685828,5.298704,20.607051
2,01005,AL,Barbour County,4,0,25259,55.390158,50.314607,24.328980,1.287412,8.624186,5.225860,19.007087
3,01007,AL,Bibb County,4,0,22412,25.950384,40.404762,19.461917,0.324721,9.706819,5.336427,16.036052
4,01009,AL,Blount County,4,0,58884,14.076829,33.465357,16.351923,1.582160,6.023723,5.887847,17.974322
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2574,56037,WY,Sweetwater County,8,0,42459,21.705645,22.398131,7.294363,1.976946,6.713718,6.257802,12.694599
2575,56039,WY,Teton County,8,0,23319,20.073760,21.523236,3.969647,4.889309,2.402089,4.579956,15.476650
2576,56041,WY,Uinta County,8,0,20514,12.971629,25.687078,6.430892,1.641694,3.467517,6.887979,14.731403
2577,56043,WY,Washakie County,8,0,6845,20.525931,23.853073,6.220807,0.340832,2.671862,5.084003,21.183346


In [8]:
save_df.to_csv('data/EJScreen_DEMO23.csv')